In [1]:
import polars as pl

In [18]:
df = pl.read_excel("dp_related.xlsx",sheet_name="GAIN_MAPPING")

In [19]:
df

id,Subsys_Category,boa_val,boa_code
i64,str,str,str
1,"""KU DA""","""FGM 0,FGM 0.5,FGM 1,FGM 1.5,FG…","""000000,000001,000002,000003,00…"
2,"""C DA""","""FGM 0,FGM 2,FGM 4,FGM 6,FGM 8,…","""00000F,00001E,00000D,00000C,00…"
3,"""UHF DA""","""FGM 0,FGM 1,FGM 2,FGM 3,FGM 4,…","""000000,000002,000004,000006,00…"
4,"""S DA""","""FGM 0,FGM 1,FGM 2,FGM 3,FGM 4,…","""000000,000002,000004,000006,00…"
5,"""S SSPA""","""FGM 0,FGM 1,FGM 2,FGM 3,FGM 4,…","""000000,000002,000004,000006,00…"
…,…,…,…
17,"""Return link Power Channelizer …","""Main-1,Main-2,Redt-1,Redt-2""","""00,01,10,11"""
18,"""Return link Power channel sele…","""Port-1,Port-2,Port-3,Port-4,Po…","""000,001,010,011,100,101,110,11…"
19,"""Return link Power Threshold fo…","""1(-38 dBm),2(-34.98 dBm),3(-33…","""0000000000000001,0000000000000…"


In [32]:
import polars as pl


# Clean up Subsys_Category values
df = df.with_columns(
    pl.col("Subsys_Category").str.strip_chars('"').str.replace_all(" ", "_")
)

# Explode each row to multiple rows for BOA values and codes
data = {}
for row in df.iter_rows(named=True):
    category = row["Subsys_Category"]
    val_list = row["boa_val"].split(",")
    code_list = row["boa_code"].split(",")
    data[f"{category}_VALUE"]= val_list
    data[f"{category}_CODE"] = code_list
            
# Step 1: Find the maximum list length
max_len = max(len(v) for v in data.values())

# Step 2: Pad all lists to the same length
padded_data = {
    k: v + [None] * (max_len - len(v))  # pad with None
    for k, v in data.items()
}

# Final expanded dataframe
expanded_df = pl.DataFrame(padded_data)
print(expanded_df)



shape: (1_972, 54)
┌─────────────┬────────────┬────────────┬───────────┬───┬───────────────┬───────────────┬───────────────┬──────────────┐
│ KU_DA_VALUE ┆ KU_DA_CODE ┆ C_DA_VALUE ┆ C_DA_CODE ┆ … ┆ Return_link_G ┆ Return_link_G ┆ Return_link_c ┆ Return_link_ │
│ ---         ┆ ---        ┆ ---        ┆ ---       ┆   ┆ ain_control_c ┆ ain_control_c ┆ hannel_Synthe ┆ channel_Synt │
│ str         ┆ str        ┆ str        ┆ str       ┆   ┆ ommu…         ┆ ommu…         ┆ size…         ┆ hesize…      │
│             ┆            ┆            ┆           ┆   ┆ ---           ┆ ---           ┆ ---           ┆ ---          │
│             ┆            ┆            ┆           ┆   ┆ str           ┆ str           ┆ str           ┆ str          │
╞═════════════╪════════════╪════════════╪═══════════╪═══╪═══════════════╪═══════════════╪═══════════════╪══════════════╡
│ FGM 0       ┆ 000000     ┆ FGM 0      ┆ 00000F    ┆ … ┆ 5 dB          ┆ 0000          ┆ 1(290.001)    ┆ 00000000001  │
│ FGM 0.5    

In [29]:
expanded_df.select(pl.col(["KU_DA_BOA_VALUE","KU_DA_BOA_CODE"])).drop_nulls()

KU_DA_BOA_VALUE,KU_DA_BOA_CODE
str,str
"""FGM 0""","""000000"""
"""FGM 0.5""","""000001"""
"""FGM 1""","""000002"""
"""FGM 1.5""","""000003"""
"""FGM 2""","""000004"""
…,…
"""ALC 13""","""001A00"""
"""ALC 13.5""","""001B00"""
"""ALC 14""","""001C00"""


In [33]:
expanded_df.to_pandas().to_excel("op.xlsx")